## Training notebook

In [1]:
import gym
import highway_env

from stable_baselines3 import PPO
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

%load_ext tensorboard
from datetime import datetime
import os
from ipywidgets import interact
import ipywidgets as widgets
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import CheckpointCallback, EvalCallback
from stable_baselines3.common.env_checker import check_env
from highway_env.tb_callback import TensorboardCallback

from typing import Callable

### Select operating system

In [2]:
os_id = widgets.Text()
def f(desired_os):
    os_id.value = str(desired_os)
interact(f, desired_os=['UBUNTU_PIGO', 'UBUNTU_DELL'])

interactive(children=(Dropdown(description='desired_os', options=('UBUNTU_PIGO', 'UBUNTU_DELL'), value='UBUNTU…

<function __main__.f(desired_os)>

In [3]:
if os_id.value == 'UBUNTU_DELL':
    OUTPUT_PATH = '/home/pighetti/HighwayDRL/training_output'
elif os_id.value == 'UBUNTU_PIGO':
    OUTPUT_PATH = '/home/pigo/HighwayDRL/training_output'

### Select environment

In [4]:
env_id = widgets.Text()
def f(input_env):
    env_id.value = str(input_env)
interact(f, input_env=['multipleO-dm-env-v0','highway-v0','dm-env-v0','acc-dm-env-v0','jam-dm-env-v0','overtaken-dm-env-v0','singleO-dm-env-v0'])

interactive(children=(Dropdown(description='input_env', options=('multipleO-dm-env-v0', 'highway-v0', 'dm-env-…

<function __main__.f(input_env)>

In [5]:
total_timesteps = 8e4
# Create and wrap the environment
env = gym.make(env_id.value)
env = Monitor(env, filename='../../training_output/log') 
check_env(env)
env.configure({
    "training_total_timesteps": total_timesteps
})

# Save a checkpoint every 5000 steps
checkpoint_callback = CheckpointCallback(save_freq=5000, save_path=f'{OUTPUT_PATH}/checkpoints',
                                         name_prefix='dm_rl_callback_CONS')

eval_callback = EvalCallback(env, best_model_save_path=f'{OUTPUT_PATH}/eval_logs',
                             log_path=f'{OUTPUT_PATH}/eval_logs', eval_freq=1000,
                             deterministic=True, render=False)

tb_callback = TensorboardCallback()

/home/pigo/miniconda3/envs/highway-env/lib/python3.8/site-packages/stable_baselines3/common/env_checker.py:190: UserWarning: Your observation  has an unconventional shape (neither an image, nor a 1D vector). We recommend you to flatten the observation to have only a 1D vector or use a custom policy to properly process the data.
  warnings.warn(


In [6]:
def linear_schedule(initial_value: float) -> Callable[[float], float]:
    """
    Linear learning rate schedule.

    :param initial_value: Initial learning rate.
    :return: schedule that computes
      current learning rate depending on remaining progress
    """
    def func(progress_remaining: float) -> float:
        """
        Progress will decrease from 1 (beginning) to 0.

        :param progress_remaining:
        :return: current learning rate
        """
        return progress_remaining * initial_value

    return func

In [7]:
ppo_ilr = 3.5e-4
# PPO parameters
model = PPO("MlpPolicy",
        env,
        gamma=0.997,
        batch_size=128,
        n_steps=4096,
        n_epochs=10,
        ent_coef=0.1,
        verbose=0,
        learning_rate=ppo_ilr,
        tensorboard_log=f'{OUTPUT_PATH}/logdir'
        )

In [8]:
try:
    # Train the agent for n steps
    model.learn(total_timesteps=int(total_timesteps), callback=[checkpoint_callback, tb_callback], progress_bar=True)
finally:
    # Save the trained agent
    model.save(f'{OUTPUT_PATH}/models/ppo_2lane_{str(os_id.value)}_{total_timesteps:.1E}_{env_id.value}_{datetime.now().strftime("%d-%m_%H-%M-%S")}.zip')

Output()

KeyboardInterrupt: 

## Continued learning

In [ ]:
env_cl_id = 'dm-env-v0'
env_cl = gym.make(env_cl_id)


OUTPUT_PATH = '/home/pigo/HighwayDRL/training_output'

In [ ]:
model_cl = PPO.load(f"{OUTPUT_PATH}/models/ppo_CONSERVATIVE_UBUNTU_PIGO_1.0E+06_multipleO-dm-env-v0_08-04_02-56-56", env=env_cl, tensorboard_log=f"{OUTPUT_PATH}/logdir")

Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [ ]:
class TensorboardCallback(BaseCallback):
    def __init__(self, verbose=1):
        super(TensorboardCallback, self).__init__(verbose)
        self.coll_rew = 0
        self.rml_rew = 0
        self.high_speed_rew = 0
        self.current_steps = 0

    def _on_step(self) -> bool:        
        self.rml_rew += self.training_env.get_attr('rml_reward')[0]
        self.high_speed_rew += self.training_env.get_attr('high_speed_reward')[0]
            
        if self.locals["dones"][0]:
            self.coll_rew += -self.training_env.get_attr('collision_reward')[0]
            self.logger.record("eval/coll_rew", self.coll_rew)
            self.logger.record("episode/RML_rew", self.rml_rew / self.current_steps)
            self.logger.record("episode/high_speed_rew", self.high_speed_rew / self.current_steps)
            self.rml_rew = self.high_speed_rew = self.current_steps = 0
        else:
            self.current_steps = self.training_env.get_attr('steps')[0]
        return True

# Save a checkpoint every 1000 steps
checkpoint_callback = CheckpointCallback(save_freq=5000, save_path=f'{OUTPUT_PATH}/checkpoints',
                                         name_prefix='dm_rl_callback')

eval_callback = EvalCallback(env_cl, best_model_save_path=f'{OUTPUT_PATH}/eval_logs',
                             log_path=f'{OUTPUT_PATH}/eval_logs', eval_freq=1000,
                             deterministic=True, render=False)

In [ ]:
%%capture output
new_timesteps = 1e6
try:
    model_cl.learn(total_timesteps=int(new_timesteps), callback=[checkpoint_callback, eval_callback, TensorboardCallback()], reset_num_timesteps=False)
finally:
    model_cl.save(f'{OUTPUT_PATH}/models/ppo_conservative_3lane_{env_cl_id}_{datetime.now().strftime("%d-%m_%H-%M-%S")}.zip')

KeyboardInterrupt: 